## Environment & Settings

import os
from dotenv import load_dotenv


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

load_dotenv(override=True)

api_key = os.getenv("GOOGLE_API_KEY")
print("API key Loaded?:", bool(api_key))


API key Loaded?: True


### Vector Store & Embeddings

In [2]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Initialize embeddings model
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)
# Create FAISS database from scratch
vectorstore = FAISS.from_texts(
    texts=["Employees get 20 days of paid annual leave.", "Remote work is allowed 2 days in a week."],
    embedding = embeddings,
    metadatas = [{"source": "HR Policy"}, {"source": "Remote Policy"}]
)

# Test raw search output
results = vectorstore.similarity_search("How many remote days", k=1)
print("Type of result:", type(results))
print("Document content:", results[0].page_content)
print("Document metadata:", results[0].metadata)
print(results)

/var/folders/xt/vgxlrb7n5b3_9v_5s_0c0dqh0000gn/T/ipykernel_1833/1786699153.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Type of result: <class 'list'>
Document content: Remote work is allowed 2 days in a week.
Document metadata: {'source': 'Remote Policy'}
[Document(id='f052c62f-07a0-48e3-a0df-776efd61d618', metadata={'source': 'Remote Policy'}, page_content='Remote work is allowed 2 days in a week.')]


In [3]:
from langchain_core.tools import tool

@tool
def search_internal_knowledge(query: str) -> str:
    """Searches the company internal document vector store."""
    docs = vectorstore.similarity_search(query, k =1)
    return docs[0].page_content if docs else "No results found."

# Run the tool function directly like standard Python
tool_result = search_internal_knowledge.invoke("annual leave")
print(f"Tool Output: {tool_result}")

Tool Output: Employees get 20 days of paid annual leave.


### Model & Tool Binding

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash", temperature=0)
model_with_tools = llm.bind_tools([search_internal_knowledge])

# Send a message to see if LLM generates a tool call request
response = model_with_tools.invoke([HumanMessage(content= "How many remork work days are allowed?")])
print("This is how response looks like: ",response)
print("LLm Response Type:", type(response))
print("Tool Calls Requested by LLM:", response.tool_calls)

GoogleRateLimitError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 3.566608189s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'q

### LangGraph Execution

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.prebuilt import tools_condition,ToolNode

builder = StateGraph(MessagesState)
builder.add_node("assistant", lambda state: {"messages": [model_with_tools.invoke(state["messages"])]})
builder.add_node("tools", ToolNode([search_internal_knowledge]))

builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")

graph = builder.compile()

print(graph)
output = graph.invoke({"messages":[HumanMessage(content="How many weeks")]})
print(output)
output['messages'][-1].content[0]['text']
